In [1]:
# Import necessary libraries
# No changes needed in this cell
from openai import OpenAI  # For accessing the OpenAI API
from enum import Enum
import json
from pydantic import BaseModel, Field  # For structured data validation
from typing import List, Literal, Optional

In [2]:
import os

In [ ]:
# Set up LLM credentials
os.environ["OPENAI_API_KEY"] = "A"

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [4]:
# Define helper functions
# No changes needed in this cell


class OpenAIModels(str, Enum):
    GPT_4O_MINI = "gpt-4o-mini"
    GPT_41_MINI = "gpt-4.1-mini"
    GPT_41_NANO = "gpt-4.1-nano"


MODEL = OpenAIModels.GPT_41_NANO


def get_completion(messages=None, system_prompt=None, user_prompt=None, model=MODEL):
    """
    Function to get a completion from the OpenAI API.
    Args:
        system_prompt: The system prompt
        user_prompt: The user prompt
        model: The model to use (default is gpt-4.1-mini)
    Returns:
        The completion text
    """

    messages = list(messages)
    if system_prompt:
        messages.insert(0, {"role": "system", "content": system_prompt})
    if user_prompt:
        messages.append({"role": "user", "content": user_prompt})
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7,
    )
    return response.choices[0].message.content

In [5]:
sample_fnols = [
    """
    Claim ID: C001
    Customer: John Smith
    Vehicle: 2018 Toyota Camry
    Incident: While driving on the highway, a rock hit my windshield and caused a small chip
    about the size of a quarter. No other damage was observed.
    """,
    """
    Claim ID: C002
    Customer: Sarah Johnson
    Vehicle: 2020 Honda Civic
    Incident: I was parked at the grocery store and returned to find someone had hit my car and
    dented the rear bumper and taillight. The taillight is broken and the bumper has a large dent.
    """,
    """
    Claim ID: C003
    Customer: Michael Rodriguez
    Vehicle: 2022 Ford F-150
    Incident: I was involved in a serious collision at an intersection. The front of my truck is
    severely damaged, including the hood, bumper, radiator, and engine compartment. The airbags
    deployed and the vehicle is not drivable.
    """,
    """
    Claim ID: C004
    Customer: Emma Williams
    Vehicle: 2019 Subaru Outback
    Incident: My car was damaged in a hailstorm. There are multiple dents on the hood, roof, and
    trunk. The side mirrors were also damaged and one window has a small crack.
    """,
    """
    Claim ID: C005
    Customer: David Brown
    Vehicle: 2021 Tesla Model 3
    Incident: Someone keyed my car in the parking lot. There are deep scratches along both doors
    on the driver's side.
    """,
]

In [6]:
# Define a system prompt for information extraction according to the provided ClaimInformation class
# TODO: Complete the prompt by replacing the parts marked with **********


class ClaimInformation(BaseModel):
    claim_id: str = Field(..., min_length=2, max_length=10)
    name: str = Field(..., min_length=2, max_length=100)
    vehicle: str = Field(..., min_length=2, max_length=100)
    loss_desc: str = Field(..., min_length=10, max_length=500)
    damage_area: List[
        Literal[
            "windshield",
            "front",
            "rear",
            "side",
            "roof",
            "hood",
            "door",
            "bumper",
            "fender",
            "quarter panel",
            "trunk",
            "glass",
        ]
    ] = Field(..., min_items=1)


info_extraction_system_prompt = """
You are an auto insurance claim processing assistant. Your task is to extract key information from First Notice of Loss (FNOL) reports.

Format your response as a valid JSON object with the following keys:
- claim_id (str): The claim ID
- name (str): The customer's full name
- vehicle (str): The vehicle make, model, and year
- loss_desc (str): A concise description of the incident
- damage_area (list[str]): A list of damaged areas on the vehicle (at least one of:
    - windshield
    - front
    - rear
    - side
    - roof
    - hood
    - door
    - bumper
    - fender
    - quarter panel
    - trunk
    - glass

For damage_area, only use items from the list above.

Only respond with the JSON object, nothing else.
"""

/tmp/ipykernel_1070/2337826049.py:25: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  ] = Field(..., min_items=1)


In [7]:
# Define a gate check function and claim extraction function

def gate1_validate_claim_info(claim_info_json: str) -> ClaimInformation:
    """
    Gate 1: Validates claim information extracted from FNOL text.
    Returns validated ClaimInformation object or raises validation error.
    """
    try:
        # Parse the JSON string
        claim_info_dict = json.loads(claim_info_json)
        # Validate with Pydantic model
        validated_info = ClaimInformation(**claim_info_dict)
        return validated_info
    except Exception as e:
        raise ValueError(f"Gate 1 validation failed: {str(e)}")


def extract_claim_info(fnol_text):
    """
    Stage 1: Extract structured information from FNOL text
    """
    messages = [
        {"role": "system", "content": info_extraction_system_prompt},
        {"role": "user", "content": fnol_text},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the extracted information
    try:
        validated_info = gate1_validate_claim_info(response)
        return validated_info
    except ValueError as e:
        print(f"Gate 1 failed: {e}")
        return None

In [8]:
# Run the claim extraction function on the sample FNOLs

extracted_claim_info_items = [
    extract_claim_info(fnol_text) for fnol_text in sample_fnols
]
extracted_claim_info_items

Gate 1 failed: Gate 1 validation failed: 1 validation error for ClaimInformation
damage_area.2
  Input should be 'windshield', 'front', 'rear', 'side', 'roof', 'hood', 'door', 'bumper', 'fender', 'quarter panel', 'trunk' or 'glass' [type=literal_error, input_value='tail light', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


[ClaimInformation(claim_id='C001', name='John Smith', vehicle='2018 Toyota Camry', loss_desc='A rock hit the windshield causing a small chip.', damage_area=['windshield']),
 None,
 ClaimInformation(claim_id='C003', name='Michael Rodriguez', vehicle='2022 Ford F-150', loss_desc='Serious collision at intersection causing severe damage to front of the truck, including hood, bumper, radiator, and engine compartment. Airbags deployed, vehicle not drivable.', damage_area=['hood', 'bumper']),
 ClaimInformation(claim_id='C004', name='Emma Williams', vehicle='2019 Subaru Outback', loss_desc='Car damaged in a hailstorm with dents on hood, roof, and trunk, damaged side mirrors, and a cracked window.', damage_area=['hood', 'roof', 'trunk', 'side', 'glass']),
 ClaimInformation(claim_id='C005', name='David Brown', vehicle='2021 Tesla Model 3', loss_desc="Someone keyed the car with deep scratches along both doors on the driver's side", damage_area=['door'])]

In [9]:
# Define a system prompt for severity assessment according to the provided SeverityAssessment class

class SeverityAssessment(BaseModel):
    severity: Literal["Minor", "Moderate", "Major"]
    est_cost: float = Field(..., gt=0)


severity_assessment_system_prompt = """
You are an auto insurance damage assessor. Your task is to evaluate the severity of vehicle damage and estimate repair costs.

Apply these carrier heuristics:
- Minor damage: Small dents, scratches, glass chips (cost range: $100-$1,000)
- Moderate damage: Single panel damage, bumper replacement, door damage (cost range: $1,000-$5,000)
- Major damage: Structural damage, multiple panel replacement, engine/drivetrain issues, total loss candidates (cost range: $5,000-$50,000)

Based on the claim information provided, determine:
1. Severity level (Minor, Moderate, or Major)
2. Estimated repair cost (in USD)

Format your response as a valid JSON object with the following keys:
- severity: One of "Minor", "Moderate", or "Major"
- est_cost: Numeric estimate of repair costs (e.g., 750.00)

Only respond with the JSON object, nothing else.
"""

In [10]:
# Define a gate check function and assess_severity function

def gate2_cost_range_ok(severity_json: str) -> SeverityAssessment:
    """
    Gate 2: Validates that the estimated cost is within reasonable range for the severity.
    Returns validated SeverityAssessment object or raises validation error.
    """
    try:
        # Parse the JSON string
        severity_dict = json.loads(severity_json)
        # Validate with Pydantic model
        validated_severity = SeverityAssessment(**severity_dict)

        # Check cost range based on severity
        if validated_severity.severity == "Minor" and (
            validated_severity.est_cost < 100 or validated_severity.est_cost > 1000
        ):
            raise ValueError(
                f"Minor damage should cost between $100-$1000, got ${validated_severity.est_cost}"
            )
        elif validated_severity.severity == "Moderate" and (
            validated_severity.est_cost < 1000 or validated_severity.est_cost > 5000
        ):
            raise ValueError(
                f"Moderate damage should cost between $1000-$5000, got ${validated_severity.est_cost}"
            )
        elif validated_severity.severity == "Major" and (
            validated_severity.est_cost < 5000 or validated_severity.est_cost > 50000
        ):
            raise ValueError(
                f"Major damage should cost between $5000-$50000, got ${validated_severity.est_cost}"
            )

        return validated_severity
    except Exception as e:
        raise ValueError(f"Gate 2 validation failed: {str(e)}")


def assess_severity(claim_info: ClaimInformation) -> Optional[SeverityAssessment]:
    """
    Stage 2: Assess severity based on damage description
    """

    # Convert Pydantic model to JSON string
    claim_info_json = claim_info.model_dump_json()

    messages = [
        {"role": "system", "content": severity_assessment_system_prompt},
        {"role": "user", "content": claim_info_json},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the severity assessment
    try:
        validated_severity = gate2_cost_range_ok(response)
        return validated_severity
    except ValueError as e:
        print(f"Gate 2 failed: {e}. Response: {response}")
        return None


In [12]:
severity_assessment_items = [
    assess_severity(item) for item in extracted_claim_info_items if item is not None
]

severity_assessment_items

[SeverityAssessment(severity='Minor', est_cost=200.0),
 SeverityAssessment(severity='Major', est_cost=15000.0),
 SeverityAssessment(severity='Moderate', est_cost=3000.0),
 SeverityAssessment(severity='Moderate', est_cost=2500.0)]

In [13]:
# Define a system prompt for claim routing according to the provided ClaimRouting class

class ClaimRouting(BaseModel):
    claim_id: str
    queue: Literal["glass", "fast_track", "material_damage", "total_loss"]

queue_routing_system_prompt = """
You are an auto insurance claim routing specialist. Your task is to determine the appropriate processing queue for each claim.

Use these routing rules:
- 'glass' queue: For Minor damage involving ONLY glass (windshield, windows)
- 'fast_track' queue: For other Minor damage
- 'material_damage' queue: For all Moderate damage
- 'total_loss' queue: For all Major damage

Format your response as a valid JSON object with the following keys:
- claim_id: Use the provided claim ID
- queue: One of "glass", "fast_track", "material_damage", or "total_loss"

Only respond with the JSON object, nothing else.
"""

In [14]:
# Define a gate check function and assess_severity function
# TODO: Complete the prompt by replacing the parts marked with **********


def gate3_validate_routing(routing_json: str) -> ClaimRouting:
    """
    Gate 3: Validates that the claim is routed to a valid queue.
    Returns validated ClaimRouting object or raises validation error.
    """
    try:
        # Parse the JSON string
        routing_dict = json.loads(routing_json)
        # Validate with Pydantic model
        validated_routing = ClaimRouting(**routing_dict)
        return validated_routing
    except Exception as e:
        raise ValueError(f"Gate 3 validation failed: {str(e)}")


def route_claim(
    claim_info: ClaimInformation, severity_assessment: Optional[SeverityAssessment]
) -> Optional[ClaimRouting]:
    """
    Stage 3: Route claim to appropriate queue
    """
    if severity_assessment is None:
        return None

    # Create input for the routing model
    routing_input = {
        "claim_info": claim_info.model_dump(),
        "severity_assessment": severity_assessment.model_dump(),
    }

    messages = [
        {"role": "system", "content": queue_routing_system_prompt},
        {"role": "user", "content": json.dumps(routing_input)},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the routing decision
    try:
        validated_routing = gate3_validate_routing(response)
        return validated_routing
    except ValueError as e:
        print(f"Gate 3 failed: {e}. Response: {response}")
        return None

In [16]:
# Run the routing function on the sample data

# Filter out None values from extracted_claim_info_items first to align with severity_assessment_items
valid_extracted_claim_info_items = [item for item in extracted_claim_info_items if item is not None]

routed_claim_items = [
    route_claim(claim, severity_assessment)
    for claim, severity_assessment in zip(
        valid_extracted_claim_info_items, severity_assessment_items
    )
]

routed_claim_items

[ClaimRouting(claim_id='C001', queue='glass'),
 ClaimRouting(claim_id='C003', queue='total_loss'),
 ClaimRouting(claim_id='C004', queue='material_damage'),
 ClaimRouting(claim_id='C005', queue='material_damage')]

In [17]:
import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

NumPy: 2.1.3
Pandas: 2.2.3


In [19]:
import pandas as pd

records = []
for claim, severity_assessment, routed_claim in zip(
    valid_extracted_claim_info_items, severity_assessment_items, routed_claim_items
):
    record = {}
    record.update(claim.model_dump())
    record.update(severity_assessment.model_dump())
    record.update(routed_claim.model_dump())
    records.append(record)


# Show the entire dataframe since it is not too large
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
df = pd.DataFrame(records)

df

,claim_id,name,vehicle,loss_desc,damage_area,severity,est_cost,queue
0,C001,John Smith,2018 Toyota Camry,A rock hit the windshield causing a small chip.,[windshield],Minor,200.0,glass
1,C003,Michael Rodriguez,2022 Ford F-150,"Serious collision at intersection causing severe damage to front of the truck, including hood, bumper, radiator, and engine compartment. Airbags deployed, vehicle not drivable.","[hood, bumper]",Major,15000.0,total_loss
2,C004,Emma Williams,2019 Subaru Outback,"Car damaged in a hailstorm with dents on hood, roof, and trunk, damaged side mirrors, and a cracked window.","[hood, roof, trunk, side, glass]",Moderate,3000.0,material_damage
3,C005,David Brown,2021 Tesla Model 3,Someone keyed the car with deep scratches along both doors on the driver's side,[door],Moderate,2500.0,material_damage
